# Foundation-model logits on a GPU (Google Colab)

Regenerates the **saved predictions (logits)** of TabPFN-2.5 and TabICLv2 used in the paper, in the same file layout as
the artistry repository (`logits/...`). Every correction (Platt, isotonic, the scan, ...) is then run on a CPU from these
files with `experiments/corrections.py`, so nothing else needs a GPU.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (the paper's runs and GPU timings used a free-tier T4).
2. TabPFN-2.5 weights are gated: accept the model licence on Hugging Face (`Prior-Labs/tabpfn_2_5`) and add two Colab
   secrets (key icon on the left): `HF_TOKEN` (a Hugging Face read token) and `TABPFN_TOKEN` (Prior Labs licence token).
   TabICLv2 needs no token. The secrets are read into environment variables and never printed.
3. Results are written to Google Drive (`My Drive/scan_repair_logits/`) so they survive disconnects; re-running a cell
   skips files that already exist.

**Time on a T4:** TabPFN ladder about 2–3 h; TabICLv2 under 1 h; discovery arm about 1 h. GPUs other than the T4 give
slightly different logits (an A100 rerun of TabICLv2 changed test log-loss by at most 7×10⁻⁵ per run).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU: switch the runtime to a GPU'

In [ ]:
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/scan_repair_logits'      # mirrors the artistry repository layout
os.makedirs(OUT, exist_ok=True)
!rm -rf /content/scan-repair-foundation-models
!git clone -q https://github.com/kayvinsoo7/scan-repair-foundation-models /content/scan-repair-foundation-models
%cd /content/scan-repair-foundation-models
!pip install -q -r requirements-fm.txt

In [ ]:
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')          # not printed
os.environ['TABPFN_TOKEN'] = userdata.get('TABPFN_TOKEN')  # not printed
os.environ['TABPFN_NO_BROWSER'] = '1'
os.environ['TABPFN_BACKEND'] = 'modern'; os.environ['TABPFN_VERSION'] = 'v2.5'
os.environ['SSR_ARTISTRY'] = OUT
%cd experiments
!python check_tabpfn_setup.py

**1. TabPFN-2.5 context ladder** — contexts 500 … 10,000 and full, three datasets, 10 seeds (Sec. V, Figs. 2–4, Tables III–IV).

In [ ]:
!OUT={OUT}/logits/tabpfn25_ladder python -u base_tabpfn_ladder.py

**2. GPU timing** — seconds to score 1,000 rows at each context on this GPU (Table IV, T4 row).

In [ ]:
!OUT={OUT}/timing python -u timing_tabpfn.py

**3. TabICLv2** with the whole training split as context (Table I).

In [ ]:
!OUT={OUT}/logits/tabicl python -u base_tabicl.py

**4. Discovery arm** (n = 19,200; Table II, Fig. 1): both foundation models, then the 2,400-row variant.

In [ ]:
for m in ['tabicl', 'tabpfn']:
    !MODEL={m} OUT={OUT}/logits/discovery/{m} python -u base_fm_discovery.py
    !python make_discovery_small.py {OUT}/logits/discovery/{m}

In [ ]:
import glob
for d in ['logits/tabpfn25_ladder', 'logits/tabicl', 'logits/discovery/tabicl', 'logits/discovery/tabpfn']:
    print(d, len(glob.glob(f'{OUT}/{d}/*.npz')), 'files')   # expected 240, 30, 40, 40
print('Next (on any CPU): python experiments/corrections.py <folder> <output>, then compare with tests/compare_corrections.py')